##1. Lab Overview
This practical is designed for Week 2 of DSA 2050. It focuses on data acquisition and SQL, with particular
attention to relational keys, aggregation grain, SELECT, WHERE, GROUP BY, JOIN, missing or duplicate keys, and
reconciliation after integration.

## Learning Outcomes

- Explain the grain of customer and order datasets.
- Detect duplicate and unmatched relational keys before joining data.
- Create and query a SQLite database from pandas DataFrames.
- Use `SELECT`, `WHERE`, `GROUP BY`, `COUNT`, `SUM`, and `AVG`.
- Join customer and order tables correctly using relational keys.
- Reconcile row counts and sales totals before and after joins.
- Merge a JSON lookup table and calculate regional KPIs.
- Write short evidence-based business interpretations.

## Scenario

You are a junior data analyst working for a retail company. Management wants to understand customer and sales performance across customer segments and regions. The data is stored in separate sources and contains deliberate quality issues. Your job is to combine the data carefully, investigate the problems, and produce trustworthy KPIs.

> **Important:** Do not delete or “fix” records before you have investigated and documented the issue. The purpose of the lab is to see how relational data problems affect analytical results.


## What You Will Build

1. Create three small source datasets: `customers.csv`, `orders.csv`, and `regions.json`.
2. Load the customer and order data into SQLite.
3. Write SQL queries to filter, aggregate, and summarize sales.
4. Join customers to orders and inspect the effect of duplicate keys.
5. Correct the duplicate-key issue and confirm totals are preserved.
6. Merge the region lookup and produce regional sales KPIs.
7. Push the completed notebook and source files to GitHub.

#2. Part A – Create the Source Data

In [1]:
import pandas as pd
import sqlite3
import json

#– Create the Orders Data
customers = pd.DataFrame({
    "CustomerID": ["C001","C002","C003","C004","C004","C005","C006","C007","C008","C009","C010"],
    "CustomerName": ["Amina","Brian","Carol","David","David","Esther","Faith","George","Halima","Ian","Jane"],
    "Segment": ["Retail","Corporate","Retail","SME","SME","Corporate","SME","Retail","Corporate","Retail","SME"],
    "RegionCode": ["NRB","MSA","NRB","KSM","KSM","NKR","NRB","MSA","KSM","NKR","NRB"]
})
customers.to_csv("customers.csv", index=False)

#Create the Orders Data
orders = pd.DataFrame({
    "OrderID": ["O001","O002","O003","O004","O005","O006","O007","O008","O009","O010","O011","O012","O013","O014","O015"],
    "CustomerID": ["C001","C002","C001","C003","C004","C005","C006","C002","C007","C008","C009","C010","C006","C003","C999"],
    "OrderAmount": [4500,12500,3200,6800,9100,15000,7600,8300,2500,11000,5400,6200,4300,7200,9900],
    "ProductCategory": ["Electronics","Office","Home","Electronics","Office","Electronics","Home","Office","Home","Electronics","Office","Home","Electronics","Office","Electronics"]
})
orders.to_csv("orders.csv", index=False)

#Create the JSON Region Lookup
regions = [
    {"RegionCode": "NRB", "RegionName": "Nairobi"},
    {"RegionCode": "MSA", "RegionName": "Mombasa"},
    {"RegionCode": "KSM", "RegionName": "Kisumu"},
    {"RegionCode": "NKR", "RegionName": "Nakuru"}
]
with open("regions.json", "w") as file:
    json.dump(regions, file, indent=4)

In [2]:
customers

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM
5,C005,Esther,Corporate,NKR
6,C006,Faith,SME,NRB
7,C007,George,Retail,MSA
8,C008,Halima,Corporate,KSM
9,C009,Ian,Retail,NKR


In [3]:
orders

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office
5,O006,C005,15000,Electronics
6,O007,C006,7600,Home
7,O008,C002,8300,Office
8,O009,C007,2500,Home
9,O010,C008,11000,Electronics


##Part B – Inspect the Sources

In [4]:
#Reload the CSV files so that your analysis begins from the saved source files rather than the in-memory DataFrames.
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")

customers.head()


,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


In [5]:
orders.head()


,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office


In [6]:
customers.shape


(11, 4)

In [7]:
orders.shape

(15, 4)

### **Q1 What does one row represent in customers.csv?**

One row in `customers.csv` represents one customer record.

### **Q2  What does one row represent in orders.csv?**

One row in `orders.csv` represents one order/transaction.

### **Q3  Are the two datasets at the same aggregation grain? Explain briefly.**

No the are not at the same aggregation grain because Customers grain = customer. Orders grain = order. They are different grains.



 ## Part C – Detect Key Problems

In [22]:
print("Duplicate CustomerID count:", customers["CustomerID"].duplicated().sum())
display(customers[customers["CustomerID"].duplicated(keep=False)])
print("Unmatched CustomerIDs:", set(orders["CustomerID"]) - set(customers["CustomerID"]))

Duplicate CustomerID count: 1


,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


Unmatched CustomerIDs: {'C999'}


### **Q4. Why can a duplicate CustomerID be dangerous when joining customer and order tables**
A duplicate `CustomerID` can multiply matching order rows when you join, inflating row counts, totals, averages, and KPIs.

### **Q5. Which CustomerID appears in orders but not in customers**?
 `C999` appears in orders but not customers.


### **Q6. Why should an unmatched record be investigated rather than silently deleted?**
 Unmatched records should be investigated because they may indicate missing data, data entry errors, new customers not loaded, or valid guest orders. Deleting them silently loses sales and biases results.

 *Do not fix the data yet. You need the duplicate and unmatched records intact so that you can observe how they affect a
JOIN.*

## Part D – Create the SQLite Database

*SQLite allows us to practise relational querying locally without needing a separate database server.*

In [9]:
#Test the Database
conn = sqlite3.connect("retail_lab.db")

customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)

query = """
SELECT *
FROM customers
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


##Part E – SELECT and WHERE

In [10]:
#Task E1 – Basic SELECT
query = """
SELECT OrderID, CustomerID, OrderAmount
FROM orders;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount
0,O001,C001,4500
1,O002,C002,12500
2,O003,C001,3200
3,O004,C003,6800
4,O005,C004,9100
5,O006,C005,15000
6,O007,C006,7600
7,O008,C002,8300
8,O009,C007,2500
9,O010,C008,11000


### # Task E2 – Filter with WHERE

1. Write a query that returns only customers in the Retail segment.

2. Write a query that returns orders with OrderAmount greater than KSh 8,000.

3. Write a query that returns only Electronics orders.

In [11]:
#Retail customers:
retail_customers_query = """
SELECT *
FROM customers
WHERE Segment = 'Retail';
"""
print("Retail Customers:")
print(pd.read_sql_query(retail_customers_query, conn))

#Orders over KSh 8,000:
large_orders_query = """
SELECT *
FROM orders
WHERE OrderAmount > 8000;
"""
print("\nOrders over KSh 8,000:")
print(pd.read_sql_query(large_orders_query, conn))

#Electronics orders:
electronics_orders_query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics';
"""
print("\nElectronics Orders:")
print(pd.read_sql_query(electronics_orders_query, conn))

#Electronics over KSh 8,000:
electronics_large_orders_query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics'
  AND OrderAmount > 8000;
"""
print("\nElectronics Orders over KSh 8,000:")
print(pd.read_sql_query(electronics_large_orders_query, conn))

Retail Customers:
  CustomerID CustomerName Segment RegionCode
0       C001        Amina  Retail        NRB
1       C003        Carol  Retail        NRB
2       C007       George  Retail        MSA
3       C009          Ian  Retail        NKR

Orders over KSh 8,000:
  OrderID CustomerID  OrderAmount ProductCategory
0    O002       C002        12500          Office
1    O005       C004         9100          Office
2    O006       C005        15000     Electronics
3    O008       C002         8300          Office
4    O010       C008        11000     Electronics
5    O015       C999         9900     Electronics

Electronics Orders:
  OrderID CustomerID  OrderAmount ProductCategory
0    O001       C001         4500     Electronics
1    O004       C003         6800     Electronics
2    O006       C005        15000     Electronics
3    O010       C008        11000     Electronics
4    O013       C006         4300     Electronics
5    O015       C999         9900     Electronics

Electronics

##Part F – GROUP BY and Aggregate Functions

Management usually needs summarized information rather than raw transaction rows. Use SQL aggregates to
calculate category-level KPIs.

In [12]:
#Total Sales
query_total_sales = """
SELECT
SUM(OrderAmount) AS TotalSales
FROM orders;
"""
print("Total Sales:")
print(pd.read_sql_query(query_total_sales, conn))

# Sales by Product Category
query_sales_by_category = """
SELECT
ProductCategory,
SUM(OrderAmount) AS TotalSales
FROM orders
GROUP BY ProductCategory;
"""
print("\nSales by Product Category:")
print(pd.read_sql_query(query_sales_by_category, conn))

Total Sales:
   TotalSales
0      113500

Sales by Product Category:
  ProductCategory  TotalSales
0     Electronics       51500
1            Home       19500
2          Office       42500


##Create one SQL query that returns all three KPIs for each product category:

•*Number of orders using COUNT()*

•*Total sales using SUM()*

•*Average order value using AVG()*

In [13]:
query = """
SELECT
    ProductCategory,
    COUNT(*) AS NumberOfOrders,
    SUM(OrderAmount) AS TotalSales,
    AVG(OrderAmount) AS AverageOrderValue
FROM orders
GROUP BY ProductCategory;
"""
pd.read_sql_query(query, conn)

,ProductCategory,NumberOfOrders,TotalSales,AverageOrderValue
0,Electronics,6,51500,8583.333333
1,Home,4,19500,4875.000000
2,Office,5,42500,8500.000000


Electronics leads with 6 orders, 51,500 total sales, and an average order value of 8,583.33. Home is lowest with 4 orders, 19,500 total sales, and an average order value of 4,875. Office sits in the middle with 5 orders, 42,500 total sales, and an average order value of 8,500.

## Part G – JOIN Customers and Orders

*connect order transactions to customer information. Use a LEFT JOIN so that every order remains visible,
including orders whose customer record is missing*

In [14]:
query = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    o.OrderAmount
FROM orders o
LEFT JOIN customers c
  ON o.CustomerID = c.CustomerID;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


In [27]:
print("Orders rows:", len(orders))
print("Joined rows:", len(joined))

Orders rows: 15
Joined rows: 16


###**Q8. Are the row counts equal? If not, explain exactly what caused the difference.**

 No. len(orders) = 15, len(joined) = 16. The extra row is caused by duplicate C004 in customers. Order O005 matches two customer rows, so it appears twice.

 ### **Q9. Locate the unmatched customer record in the joined output. What values appear for CustomerName and Segment?**

  For unmatched C999, CustomerName and Segment are NULL.

 ## Part H – Reconcile the Totals

In [23]:
print("Orders total:", orders["OrderAmount"].sum())
print("Joined total:", joined["OrderAmount"].sum())

Orders total: 113500
Joined total: 122600


### **Q10. Are the two sales totals equal?**

No. The totals are not equal.

- Orders total: **113,500**
- Joined total: **122,600**

### **Q11. If the total changed, which customer caused the duplication?**

Customer **C004** caused the duplication. Order **O005** matched two customer rows in `customers`, so it appeared twice in the joined output.
### **Q12. Why is total reconciliation an important validation step after a JOIN?**

Reconciliation is important because a JOIN can silently multiply or drop rows. Comparing row counts and control totals before and after the join catches key or grain problems such as duplicate keys or unmatched records.

## Part I — Correct the duplicate-key problem


In [25]:
customers_clean = customers.drop_duplicates(subset="CustomerID")
customers_clean.to_sql("customers_clean", conn, if_exists="replace", index=False)

query_clean = """
SELECT o.OrderID, o.CustomerID, c.CustomerName, c.Segment, o.OrderAmount
FROM orders o
LEFT JOIN customers_clean c
  ON o.CustomerID = c.CustomerID;
"""
clean_joined = pd.read_sql_query(query_clean, conn)

print("Original orders rows:", len(orders))
print("Clean joined rows:", len(clean_joined))
print("Original total:", orders["OrderAmount"].sum())
print("Clean joined total:", clean_joined["OrderAmount"].sum())

clean_joined

Original orders rows: 15
Clean joined rows: 15
Original total: 113500
Clean joined total: 113500


,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O006,C005,Esther,Corporate,15000
6,O007,C006,Faith,SME,7600
7,O008,C002,Brian,Corporate,8300
8,O009,C007,George,Retail,2500
9,O010,C008,Halima,Corporate,11000


### **Q13. Has the corrected JOIN preserved both the row count and the total sales value? Explain.**

Yes. The corrected JOIN preserved both the row count and the total sales value:

- Original orders rows: **15**
- Clean joined rows: **15**
- Original total: **113,500**
- Clean joined total: **113,500**

This works because `customers_clean` has one row per `CustomerID`. C999 stays unmatched, but it is preserved by the LEFT JOIN without multiplying rows.


## Part J — Customer segment KPIs


In [20]:
query = """
SELECT
    c.Segment,
    COUNT(o.OrderID) AS Orders,
    SUM(o.OrderAmount) AS TotalSales,
    AVG(o.OrderAmount) AS AverageOrderValue
FROM orders o
LEFT JOIN customers_clean c
  ON o.CustomerID = c.CustomerID
GROUP BY c.Segment;
"""
pd.read_sql_query(query, conn)

,Segment,Orders,TotalSales,AverageOrderValue
0,None,1,9900,9900.000000
1,Corporate,4,46800,11700.000000
2,Retail,6,29600,4933.333333
3,SME,4,27200,6800.000000


### **Q14. Write three sentences explaining what management can learn from the segment KPIs.**

Corporate generates the highest total sales (**46,800**) and the highest average order value (**11,700**), so it is a high-value segment. Retail places the most orders (**6**) but has a much lower average order value (**4,933.33**), suggesting volume but smaller baskets. SME is moderate (**4 orders, 27,200 total sales, 6,800 average order value**), and the unmatched order (**9,900**) needs investigation before it is trusted.

## Part K — Merge the JSON region lookup


In [21]:
regions_df = pd.read_json("regions.json")
regions_df

customers_regions = customers_clean.merge(
    regions_df,
    on="RegionCode",
    how="left"
)
customers_regions.head()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru


### **Q15. Confirm that each customer now has a readable RegionName rather than only a RegionCode.**

Yes. After the merge, each customer has a readable `RegionName`, for example:

- **NRB** = Nairobi
- **MSA** = Mombasa
- **KSM** = Kisumu
- **NKR** = Nakuru

### **Q16. If a RegionCode did not exist in regions.json, what would a left merge produce for RegionName?**

A left merge would produce **NaN / NULL** for `RegionName`.

## Final Challenge — Regional sales analysis


In [26]:
orders_regions = orders.merge(customers_regions, on="CustomerID", how="left")

regional_sales = (orders_regions
    .groupby("RegionName", dropna=False)
    .agg(NumberOfOrders=("OrderID", "count"),
         TotalSales=("OrderAmount", "sum"))
    .reset_index())

regional_sales["RegionName"] = regional_sales["RegionName"].fillna("Unmatched")
regional_sales

,RegionName,NumberOfOrders,TotalSales
0,Kisumu,2,20100
1,Mombasa,3,23300
2,Nairobi,7,39800
3,Nakuru,2,20400
4,Unmatched,1,9900


### **Q17. Which region generated the highest sales?**

Nairobi generated the highest sales: **39,800**.

### **Q18. Should management automatically conclude that this is the “best-performing” region? Explain using at least two limitations of the available data.**

No, management should not automatically call it the “best-performing” region. Limitations include:

- Revenue alone does not show profit, costs, or margins.
- The dataset is very small and contains missing or unmatched records.
- There is no time trend, market size, customer retention, or cost-to-serve data.

## Conclusion

The biggest data-quality problem was the duplicate CustomerID C004, together with the unmatched C999 order. The duplicate caused the original LEFT JOIN to multiply order O005, increasing row count from 15 to 16 and total sales from 113,500 to 122,600. Totals must be checked after joining because joins can silently inflate or lose rows when keys are not unique or not matched. After cleaning, Corporate generated the most sales at 46,800. A key limitation is that revenue alone does not measure profitability, cost, or long-term customer value.